# Generating decays

This notebook runs entirely in your browser through [Pyodide](https://pyodide.org). The first cell installs `pyevtgen` from its package index.

In [ ]:
import micropip

await micropip.install(
    "pyevtgen", index_urls="https://redeboer.github.io/pyevtgen-test/simple"
)

Creating a `Generator` reads the decay table and initializes Pythia8. In the browser, this takes about 15 seconds and prints a lot of log output. A notebook can create only one `Generator`, so restart the kernel to create another one.

In [ ]:
from pyevtgen import Generator

generator = Generator(seed=42)

`decay()` returns the root of a decay tree. Change the particle name to decay something else, for instance `'B+'` or `'J/psi'`:

In [ ]:
def print_tree(particle, indent: int = 0) -> None:
    print(f"{' ' * indent}{particle.name:<10} E = {particle.p4[0]:.3f} GeV")
    for daughter in particle.daughters:
        print_tree(daughter, indent + 2)


print_tree(generator.decay("D*+"))

`generate()` produces many decays. Here we count the most frequent final states of the $D^0$:

In [ ]:
from collections import Counter

events = generator.generate("D0", n=5_000)
final_states = Counter(" ".join(p.name for p in e.final_state()) for e in events)
final_states.most_common(5)

Particle properties come from the particle table:

In [ ]:
generator.properties("D0")

The momentum spectra of the final-state particles show the variety of decays in the table:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

spectra = {name: [] for name in ["pi+", "pi-", "K-", "gamma"]}
for event in events:
    for particle in event.final_state():
        if particle.name in spectra:
            spectra[particle.name].append(np.linalg.norm(particle.p4[1:]))

fig, ax = plt.subplots(figsize=(6, 3.5))
for name, momenta in spectra.items():
    ax.hist(momenta, bins=50, range=(0, 1), histtype="step", label=name)
ax.set_xlabel("$|p|$ [GeV]")
ax.set_yscale("log")
ax.legend()
plt.show()